# Experiment: APS Failure Classification (Revised)

## Dataset Choice (Original size in 50k-100k range)
- **Dataset:** APS Failure at Scania Trucks (OpenML: `APSFailure`)
- **Original size:** **76,000 rows**, 170 features
- **Task:** Binary classification (`pos` = component failure, `neg` = non-failure)

This satisfies your requirement to use a dataset whose **original size** is between **50k and 100k**.


In [ ]:
# Optional dependencies if needed:
# %pip install pandas scikit-learn matplotlib seaborn


In [ ]:
from __future__ import annotations

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

SEED = 42
CACHE_DIR = 'output-revised/data-cache-openml'
SEED


## 1) Load Dataset


In [ ]:
X_raw, y_raw = fetch_openml(
    name='APSFailure',
    version=1,
    return_X_y=True,
    as_frame=True,
    data_home=CACHE_DIR,
)

print('Original dataset shape:', X_raw.shape)
print('Target distribution:')
print(y_raw.value_counts())


## 2) Data Preparation

- Convert features to numeric (`errors='coerce'`) so invalid strings become missing values.
- Convert target to binary: `pos -> 1`, `neg -> 0`.
- Use a **50,000-row stratified working sample** for practical runtime.

Note: original dataset remains 76,000 rows (within requirement).


In [ ]:
X = X_raw.apply(pd.to_numeric, errors='coerce')
y = (y_raw == 'pos').astype(int)

# Runtime-friendly working sample while preserving class ratio
sample_n = 50000
df = pd.concat([X, y.rename('target')], axis=1)
parts = []
for cls, grp in df.groupby('target'):
    k = max(1, int(round(len(grp) * sample_n / len(df))))
    parts.append(grp.sample(n=min(k, len(grp)), random_state=SEED))

df_sample = pd.concat(parts, ignore_index=True)
X_s = df_sample.drop(columns=['target'])
y_s = df_sample['target']

print('Working sample shape:', X_s.shape)
print('Working target distribution:')
print(y_s.value_counts())


In [ ]:
ax = y_s.value_counts().rename({0:'neg',1:'pos'}).plot(kind='bar', figsize=(5,4), title='Class Distribution (Working Sample)')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


## 3) Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_s,
    y_s,
    test_size=0.2,
    stratify=y_s,
    random_state=SEED,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)


## 4) Model

Primary objective: detect failures (`pos` class), so we prioritize **recall for positive class**.

Model pipeline:
- `SimpleImputer(median)` for missing values
- `StandardScaler` for numeric stability
- `LogisticRegression(class_weight='balanced')` to handle class imbalance


In [ ]:
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        solver='liblinear',
        class_weight='balanced',
        max_iter=300,
        random_state=SEED,
    )),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

metrics = pd.Series({
    'test_accuracy': accuracy_score(y_test, y_pred),
    'test_f1_macro': f1_score(y_test, y_pred, average='macro'),
    'test_recall_pos': recall_score(y_test, y_pred, pos_label=1),
    'test_precision_pos': precision_score(y_test, y_pred, pos_label=1),
})

metrics


In [ ]:
print(classification_report(y_test, y_pred, target_names=['neg','pos']))

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(ax=ax, colorbar=False)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()


## 5) Observed Results from Validated Run

Using this exact setup, observed metrics were:
- `test_accuracy`: **0.9722**
- `test_f1_macro`: **0.7650**
- `test_recall_pos`: **0.9171**
- `test_precision_pos`: **0.3869**

Interpretation:
- High recall for failures (good for catching most true failure cases).
- Lower precision means more false alarms, which is a common trade-off when recall is prioritized.


## 6) Submission Guidance

For your report:
- Clearly state why APS Failure is a binary classification problem.
- Mention original dataset size (76,000) to satisfy requirement.
- Explain imbalance and why `class_weight='balanced'` and recall-oriented evaluation were used.
- Discuss trade-off between high recall and lower precision in operational terms (maintenance cost vs missed failures).
- Propose next steps: threshold tuning, model comparison, and cost-sensitive optimization.
